# 커스텀 감사 미들웨어

에이전트 전체와 도구 호출을 개인정보 없이 어떤 이벤트로 기록할까요?


## 실행 환경 준비

실습 파일은 `labs/part6/ch22-middleware/notebooks/06-커스텀-감사-미들웨어.ipynb`입니다. Python 3.12 새 커널에서 위에서부터 실행합니다.


## 패키지 버전 확인


In [1]:
# 이 셀은 패키지 버전 확인 작업을 수행하고 확인할 값을 출력합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langgraph==1.2.11" "python-dotenv==1.2.3"
from importlib.metadata import version
print("langchain", version("langchain"))
print("langchain-openai", version("langchain-openai"))
print("langgraph", version("langgraph"))

Note: you may need to restart the kernel to use updated packages.
langchain 1.4.0
langchain-openai 1.6.0
langgraph 1.2.11


## 결정적 모델 준비

이 모델은 정해 둔 응답으로 훅 경계를 관찰하는 결정적 검사 장치입니다. `BaseChatModel` 상속 구현 자체는 이 절의 학습 대상이 아닙니다.


In [2]:
# 이 셀은 결정적 모델 준비 작업을 수행하고 확인할 값을 출력합니다.
# 정해 둔 응답으로 미들웨어 실행 순서를 재현합니다.
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.tools import tool
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
class ScriptedModel(BaseChatModel):
    replies: list[AIMessage]
    index: int = 0
    @property
    def _llm_type(self) -> str: return "scripted-model"
    def _generate(self, messages: Any, stop: Any = None, run_manager: Any = None, **kwargs: Any) -> ChatResult:
        reply = self.replies[min(self.index, len(self.replies) - 1)]
        self.index += 1
        return ChatResult(generations=[ChatGeneration(message=reply)])
    def bind_tools(self, tools: Any, *, tool_choice: Any = None, **kwargs: Any) -> "ScriptedModel":
        return self
@tool
def lookup_hours(day: str) -> str:
    """자료실 운영 시간을 요일로 조회합니다."""
    return "평일 09:00~20:00" if day == "평일" else "토요일 10:00~17:00"

## 감사 미들웨어 작성

감사 기록은 나중에 누가 어떤 실행을 했고 어느 단계가 호출됐는지 재구성하기 위한 사건 목록입니다. 이 실습의 `EVENTS`는 내용을 바꿀 수 있는 메모리 리스트이므로 위변조 방지 로그로 볼 수 없습니다. `AuditMiddleware`는 에이전트·모델·도구 경계에서 이벤트를 `EVENTS`에 추가합니다. 각 항목의 `event`는 시점을, 해시는 원문을 저장하지 않고 같은 값인지 비교할 식별값을 나타냅니다. 해시만으로 원문을 복원하지는 못하지만 짧거나 예측 가능한 비밀은 대입 공격에 취약하므로 비밀 저장 수단으로 사용하면 안 됩니다.

직렬화 셀은 이벤트를 JSON으로 바꿔 필드와 순서를 검사합니다. `sort_keys=True`는 같은 딕셔너리를 일정한 문자열로 만드는 데 도움을 주며 시간 순서를 자동 보장하지 않습니다. 모델 호출과 도구 호출의 시작·완료를 구분해야 실패가 어느 경계에서 발생했는지 알 수 있습니다.


In [3]:
# 이 셀은 감사 미들웨어 작성 작업을 수행하고 확인할 값을 출력합니다.
import hashlib
import json
EVENTS=[]
def fingerprint(text: str) -> str:
    return hashlib.sha256(b"lesson-salt:"+text.encode("utf-8")).hexdigest()[:10]
class AuditMiddleware(AgentMiddleware):
    def before_agent(self, state: Any, runtime: Any) -> None:
        text=state["messages"][0].text
        digest=fingerprint(text)
        EVENTS.append({"event":"agent_start","input_chars":len(text),
                       "fingerprint":digest})
    def wrap_tool_call(self, request: Any, handler: Any) -> Any:
        EVENTS.append({"event":"tool_start","tool":request.tool.name})
        try:
            result=handler(request)
        except Exception as error:
            EVENTS.append({"event":"tool_error","tool":request.tool.name,
                           "type":type(error).__name__}); raise
        EVENTS.append({"event":"tool_success","tool":request.tool.name})
        return result
    def after_agent(self, state: Any, runtime: Any) -> None:
        EVENTS.append({"event":"agent_end","messages":len(state["messages"])})

## 도구를 사용하는 실행


In [4]:
# 이 셀은 도구를 사용하는 실행 작업을 수행하고 확인할 값을 출력합니다.
scripted=ScriptedModel(replies=[
 AIMessage("",tool_calls=[{"name":"lookup_hours",
  "args":{"day":"평일"},"id":"audit_1","type":"tool_call"}]),
 AIMessage("09:00~20:00")])
audit_agent=create_agent(scripted,tools=[lookup_hours],middleware=[AuditMiddleware()])
audit_agent.invoke({"messages":[{"role":"user","content":"learner@example.com 운영 시간 문의"}]})
for event in EVENTS: print(event)

{'event': 'agent_start', 'input_chars': 28, 'fingerprint': '6b19ab80f2'}
{'event': 'tool_start', 'tool': 'lookup_hours'}
{'event': 'tool_success', 'tool': 'lookup_hours'}
{'event': 'agent_end', 'messages': 4}


## 비밀값 부재 검사


In [5]:
# 이 셀은 비밀값 부재 검사 작업을 수행하고 확인할 값을 출력합니다.
serialized=json.dumps(
 EVENTS,ensure_ascii=False)
print("event_order:",[e["event"] for e in EVENTS])
print("same_length_fingerprints:",fingerprint("AA"),fingerprint("BB"))
assert "learner@example.com" not in serialized
assert len("AA") == len("BB") and fingerprint("AA") != fingerprint("BB")
assert [e["event"] for e in EVENTS]==[
 "agent_start","tool_start","tool_success","agent_end"]

event_order: ['agent_start', 'tool_start', 'tool_success', 'agent_end']
same_length_fingerprints: 6c1b4c4c81 f714e7ae49


## 기록 범위 해석

운영 환경에서는 메모리 리스트 대신 접근 제어된 로그 저장소에 요청 ID, 실행 ID, 도구 이름, 성공 여부를 기록합니다. 입력 전문을 그대로 남기면 PII나 비밀값이 감사 로그에 복제될 수 있으므로 허용 필드 목록과 보존 기간을 먼저 정합니다.

관측성 로그는 성능과 오류를 분석하는 자료이고 감사 로그는 책임 있는 변경 이력을 증명하는 자료입니다. 같은 시스템에 저장할 수 있어도 접근 권한과 보존 규칙이 다릅니다. 이번 `assert`는 예상 이벤트 종류와 비밀 문자열 부재를 확인하지만 로그의 위변조 방지나 장기 보존을 검증하지 않습니다.


In [6]:
# 이 셀은 기록 범위 해석 작업을 수행하고 확인할 값을 출력합니다.
print({"events":len(EVENTS),"raw_input_saved":False,
       "tool_result_saved":False,"estimated_cost_usd":0.0})

{'events': 4, 'raw_input_saved': False, 'tool_result_saved': False, 'estimated_cost_usd': 0.0}
